# 03. Schema Intelligence — AI가 읽기 좋은 스키마
> Day 1 · 4H · 소요 약 50분

## 학습 목표

- LLM이 읽는 스키마 프롬프트(`table_info`)의 실제 모습을 확인한다.
- `COMMENT ON TABLE/COLUMN`과 FK 명시로 AI 친화적 스키마를 만든다.
- 정규화 vs LLM 친화적 비정규화(뷰)의 트레이드오프를 이해한다.
- Mermaid ERD를 작성한다.

> **선행 조건:** `01_postgres_basics.ipynb` 에서 적재한 병원 DB를 사용합니다. 이 노트북에서 추가하는 `COMMENT`·뷰는 Day 1·2 뒤 노트북(특히 `06`, `08`)에서도 계속 사용됩니다.

In [ ]:
%pip install -q psycopg2-binary sqlalchemy pandas \
    llama-index llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# Neon DSN 은 DB 접속, OpenAI 키는 LlamaIndex SQLDatabase 가 LLM 호출에 사용.
_load_secret("NEON_DSN", required=True)
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")

In [ ]:
# inspect — SQLAlchemy 의 메타데이터 조회 헬퍼.
# create_engine 으로 만든 엔진을 inspect() 에 넘기면 테이블 목록·컬럼·외래키를 코드로 읽을 수 있습니다.
from sqlalchemy import create_engine, text, inspect
import pandas as pd

engine = create_engine(os.environ["NEON_DSN"])
inspector = inspect(engine)
# get_table_names() → public 스키마의 모든 테이블 이름 리스트.
print("Tables:", inspector.get_table_names())

## LLM은 스키마를 어떻게 읽는가?

Text-to-SQL 시스템에서 LLM은 DB 스키마를 **프롬프트의 일부 텍스트**로 전달받습니다. 따라서 컬럼명·COMMENT·FK가 프롬프트에 실리는 그대로 정확도에 영향을 줍니다.

```
❌ 나쁜 스키마                    ✅ 좋은 스키마
CREATE TABLE p (                  CREATE TABLE patients (
  p_id INT PK,                      patient_id SERIAL PK,
  nm VARCHAR(100),                  name VARCHAR(100) NOT NULL,
  gen CHAR(1),                      gender CHAR(1) ...
  bd DATE,                          birth_date DATE ...
  bt VARCHAR(3)                     blood_type VARCHAR(3) ...
);                                );
                                  COMMENT ON COLUMN patients.gender
                                    IS 'M=남성, F=여성';
```

LLM이 **추측**을 줄일수록 오답 확률이 내려갑니다.

In [ ]:
# LlamaIndex 의 SQLDatabase 래퍼는 "LLM 이 보는 스키마 텍스트(table_info)" 를 자동 생성해 줍니다.
# include_tables 로 노출 범위를 제한 — 화이트리스트 효과(보안 + 토큰 절약).
from llama_index.core import SQLDatabase, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# 모든 LlamaIndex 호출이 이 두 모델을 공통으로 쓰도록 전역 설정.
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

sql_db = SQLDatabase(
    engine,
    include_tables=["patients", "doctors", "visits", "diagnoses", "departments"],
)

# get_single_table_info("이름") → CREATE TABLE 문 + 샘플 행 + COMMENT 가 합쳐진 텍스트 반환.
# 이 텍스트가 그대로 Text-to-SQL 프롬프트의 스키마 자리에 들어갑니다.
print("table_info preview for 'visits':\n")
print(sql_db.get_single_table_info("visits"))

## 좋은 네이밍 체크리스트

1. **축약 금지** — `dt → visit_date`, `amt → amount`
2. **단수/복수 통일** — 테이블은 복수(`patients`), 컬럼은 단수
3. **snake_case** — `visitDate` 대신 `visit_date`
4. **FK 컬럼명 = 참조 PK 이름** — `patient_id`, `doctor_id`
5. **의미 단위 분리** — `created_at`, `updated_at` 같은 관용어 유지

In [ ]:
# 컬럼·테이블에 한국어 설명을 붙이는 핵심 단계.
# COMMENT ON 은 PostgreSQL 의 표준 명령으로, 쓰여진 설명이 LLM 이 보는 table_info 에 그대로 실립니다.
# 같은 컬럼에 다시 COMMENT 를 걸면 기존 설명이 덮어쓰기 되어 멱등(idempotent) 적입니다.
comment_sql = """
COMMENT ON TABLE  patients             IS '환자 기본 정보';
COMMENT ON COLUMN patients.gender      IS '성별: M=남성, F=여성';
COMMENT ON COLUMN patients.blood_type  IS '혈액형: A, B, O, AB';
COMMENT ON TABLE  visits               IS '환자 진료 방문 기록';
COMMENT ON COLUMN visits.visit_type    IS 'outpatient=외래, inpatient=입원, emergency=응급';
COMMENT ON COLUMN visits.status        IS 'scheduled=예약, completed=완료, cancelled=취소, no_show=미방문';
COMMENT ON COLUMN visits.cost          IS '진료비 (원)';
COMMENT ON TABLE  diagnoses            IS '진료 시 내려진 진단 기록';
COMMENT ON COLUMN diagnoses.severity   IS 'mild=경증, moderate=중등, severe=중증';
"""
# `engine.begin()` 은 트랜잭션을 열고 블록이 정상 종료되면 자동 COMMIT — 여러 COMMENT 를 한꺼번에 적용.
with engine.begin() as conn:
    conn.execute(text(comment_sql))
print("Comments re-applied.")

In [ ]:
# COMMENT 적용 후 table_info가 어떻게 달라지는지 재확인
print(sql_db.get_single_table_info("visits"))

## FK 명시 — LLM이 JOIN 경로를 추론하게

FK가 없는 스키마에서는 LLM이 `patient_id` 컬럼이 어떤 테이블과 연결되는지 추론해야 합니다. FK를 명시하면 `table_info`에 `FOREIGN KEY (...) REFERENCES ...` 문구가 실려서 올바른 JOIN을 유도합니다.

```
-- ❌ FK 없이
CREATE TABLE visits (visit_id INT, pid INT, did INT);

-- ✅ FK 명시
CREATE TABLE visits (
    visit_id   INT PRIMARY KEY,
    patient_id INT NOT NULL REFERENCES patients(patient_id),
    doctor_id  INT NOT NULL REFERENCES doctors(doctor_id)
);
```

**`ON DELETE` 정책**: 참조 대상이 삭제될 때 처리. `CASCADE`(같이 삭제), `SET NULL`, `RESTRICT`(삭제 차단). 감사/이력 테이블은 `RESTRICT` 권장.

In [ ]:
# 외래키(FK) 메타데이터 조회 — inspector.get_foreign_keys(테이블) 가 dict 리스트로 반환.
# 각 FK 의 핵심 키:
#   constrained_columns : 이 테이블의 어떤 컬럼이 FK 인가
#   referred_table       : 참조하는 부모 테이블 이름
#   referred_columns     : 부모 테이블의 어떤 컬럼(보통 PK)
for t in ["doctors", "visits", "diagnoses"]:
    fks = inspector.get_foreign_keys(t)
    print(f"-- {t}")
    for fk in fks:
        print(f"  {fk['constrained_columns']} -> {fk['referred_table']}{fk['referred_columns']}")

## 정규화 vs LLM 친화적 비정규화

| 정규화 | 비정규화 |
|---|---|
| 데이터 무결성 우수 | 쿼리 단순 |
| 업데이트 일관성 | LLM이 이해하기 쉬움 |
| JOIN이 많음 | 중복 가능 |
| LLM이 실수할 확률 상승 | 업데이트 시 여러 곳 수정 |

**AI 관점 권장:** 원본은 **정규화 유지** + 조회용 **리포팅 뷰(`vw_*`)** 로 비정규화 제공.

In [ ]:
# 리포팅 뷰 — visits + patients + doctors + departments 를 미리 JOIN 해 둔 "넓은 단일 뷰".
# 자주 함께 조회되는 컬럼을 미리 합쳐 두면 LLM 이 JOIN 문법을 만들지 않고도 한 테이블처럼 쿼리 가능.
# CREATE OR REPLACE 는 같은 이름의 뷰가 있으면 덮어써서 셀 재실행이 안전(idempotent).
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW vw_visit_details AS
        SELECT
            v.visit_id,
            v.visit_date,
            v.visit_type,
            v.status,
            v.chief_complaint,
            v.cost,
            p.name      AS patient_name,
            p.gender    AS patient_gender,
            EXTRACT(YEAR FROM AGE(p.birth_date)) AS patient_age,   -- 계산 컬럼: 나이
            d.name      AS doctor_name,
            d.specialty AS doctor_specialty,
            dept.name   AS department_name
        FROM visits v
        JOIN patients p    ON p.patient_id = v.patient_id
        JOIN doctors  d    ON d.doctor_id  = v.doctor_id
        JOIN departments dept ON dept.department_id = d.department_id
    """))
    # 뷰에도 COMMENT 를 달면 LlamaIndex SQLDatabase 가 같이 인식.
    conn.execute(text(
        "COMMENT ON VIEW vw_visit_details IS '진료 상세 정보 (환자/의사/진료과 조인 완료)'"
    ))
print("View vw_visit_details created.")

In [ ]:
# Use the wide view — no JOIN needed in the query
# 주의: SQL 본문에 `%` 가 들어가는 경우(예: LIKE '...%') psycopg2 pyformat 충돌을 피하려면
# text() 로 감싸고 connection 을 pandas 에 넘겨야 합니다.
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT patient_name, doctor_name, department_name, visit_date, cost
        FROM vw_visit_details
        WHERE status = 'completed'
        ORDER BY visit_date DESC
        LIMIT 10
    """), conn)
print(df.to_string(index=False))

## Mermaid ERD 작성 가이드

Mermaid는 텍스트 기반 다이어그램 언어입니다. GitHub·mkdocs·Notion·VS Code 등에서 바로 렌더링되며, `mermaid.live` 에 붙여넣어 이미지를 추출할 수도 있습니다.

관계 표기:
- `||--o{` : 1 대 N
- `||--||` : 1 대 1
- `}o--o{` : N 대 N

### 병원 DB ERD (참조용)

```mermaid
erDiagram
    departments ||--o{ doctors : "has"
    doctors     ||--o{ visits  : "conducts"
    patients    ||--o{ visits  : "makes"
    visits      ||--o{ diagnoses : "results_in"

    departments {
        int department_id PK
        varchar name
        int floor
        varchar phone
    }
    doctors {
        int doctor_id PK
        varchar name
        int department_id FK
        varchar specialty
        date hire_date
        numeric salary
    }
    patients {
        int patient_id PK
        varchar name
        date birth_date
        char gender
        varchar blood_type
    }
    visits {
        int visit_id PK
        int patient_id FK
        int doctor_id FK
        date visit_date
        varchar visit_type
        varchar status
        numeric cost
    }
    diagnoses {
        int diagnosis_id PK
        int visit_id FK
        varchar icd_code
        varchar description
        varchar severity
    }
```

## 실습 과제

1. `visits` 테이블의 한 컬럼에 대해 **기존 COMMENT를 수정**한 뒤 `sql_db.get_single_table_info("visits")` 를 다시 실행해 프롬프트가 어떻게 달라지는지 확인하세요.
2. `vw_visit_details` 뷰를 이용해 **2026년 내과 외래 진료 건수**를 조회하세요.
3. 본인 프로젝트 도메인의 **ERD 초안(3테이블 이상)** 을 Mermaid로 스케치하세요. (과제 #1 제안서 자료로 사용)

In [ ]:
# TODO: 위 실습을 자유롭게 수행해 보세요.
# 여기에 구현하세요.

## 다음 노트북에서는…

스키마를 정돈했으니 이제 **`04_llamaindex_intro.ipynb`** 에서 LlamaIndex의 RAG 파이프라인 (Document → Node → Index → Query Engine)을 익힙니다. DB가 아닌 **문서** 기반 RAG의 기초를 먼저 체감한 뒤, 7H에서 다시 SQL 세계로 돌아옵니다.